In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using GeneralizedPerturbedEquilibrium: InnerLayer
using GeneralizedPerturbedEquilibrium.InnerLayer: solve_inner
using GeneralizedPerturbedEquilibrium: Tearing
using ..InnerLayer
using ..InnerLayer: InnerLayerModel, solve_inner, GGJModel, GGJParameters,
    SLAYERModel, SLAYERParameters

using Plots
using Printf
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
struct TorqueBalance{M<:InnerLayerModel,P}
    model::M
    params::P
    Q0::Float64
    P::Float64
    lu::Float64
    sval::Float64
end

function torque_balance_value(tb::TorqueBalance, Q::Number)
    Δ = solve_inner(tb.model, tb.params, ComplexF64(Q)).tearing
    alpha = 1e-2
    jxb = -imag(1.0 / (Δ+ alpha))
    return 2.0 * tb.P * (tb.Q0 - Q) / jxb, Δ
end

function torque_balance_scan(tb; Qmin=-10.0, Qmax=10.0, n=200)
    Qs = range(Qmin, Qmax; length=n)
    torque_out = [torque_balance_value(tb, q) for q in Qs]
    bal = [x[1] for x in torque_out]
    Δs = [x[2] for x in torque_out]
    #bal = [torque_balance_value(tb, q) for q in Qs]
    positive = isfinite.(bal) .& (bal .> 0.0)

    if !any(positive)
        return Qs, bal, NaN, NaN, 0.0, NaN, NaN, Δs
    end
    i = argmax(bal)
    Qpeak_ind = i
    Qs_positive = Qs[positive]
    bal_positive = bal[positive]

    i = argmax(bal_positive)
    Qpeak = Qs_positive[i]
    
    maxbal = bal_positive[i]

    br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))
    return Qs, bal, Qs_positive, bal_positive, Qpeak, br_crit, Qpeak_ind, Δs
end

In [ ]:
p = GeneralizedPerturbedEquilibrium.InnerLayer.slayer_parameters(
    n_e=1e19, t_e=1e3, t_i=1e3,
    omega=0.0, omega_e=4, omega_i=-2,
    qval=2.0, sval_r=0.5, bt=2.0, rs=1.0, R0=3.0, mu_i=2.0, zeff=1.0,
    chi_perp=1.0, chi_tor=1.0, m=2, n=1
)
p2 = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERParameters(
    ising=p.ising,
    m=p.m, n=p.n,
    tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
    P_perp=p.P_perp, P_tor=p.P_tor,
    Q_e=0.818, Q_i=-0.543, iota_e=p.iota_e,
    tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
    rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
    dr_val=p.dr_val, dgeo_val=p.dgeo_val,
    eta=p.eta, d_beta=p.d_beta,
    dc_tmp=p.dc_tmp, dc_type=p.dc_type
)

p2 = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERParameters(
    ising=p.ising,
    m=p.m, n=p.n,
    tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
    P_perp=p.P_perp, P_tor=p.P_tor,
    Q_e=2, Q_i=3, iota_e=p.iota_e,
    tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
    rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
    dr_val=p.dr_val, dgeo_val=p.dgeo_val,
    eta=p.eta, d_beta=p.d_beta,
    dc_tmp=p.dc_tmp, dc_type=p.dc_type
)

p = p2

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERModel(;),
    p,
    0.5,
    p.P_tor,
    p.lu,
    p.sval_r
)

In [ ]:
#Qs = range(-5.0, 5.0, length=200)

#vals = [torque_balance_value(tb, q) for q in Qs]
"""
n=2000:
Qpeak = 0.24512256128064033
maxbal = 0.35873684252530247
br_crit = 4.668551985584323e-5

n=200 (def):
Qpeak = 0.25125628140703515
maxbal = 0.3585219071071706
br_crit = 4.667153206033017e-5
"""
Qmi = -10
Qma = 10

Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=Qmi, Qmax=Qma, n=100)

println("Qpeak = ", Qpeak)
println("maxbal = ", maximum(bal_positive))
println("maxbal = ", maximum(bal))
println("br_crit = ", brcrit)

p1 = plot(Qs, bal, lw=2, label="balance")#,ylim=(-40, 40))
z_inds = findall(i -> bal[i]*bal[i+1] < 0, 1:length(bal)-1)
println("zero crossings at Q = ", Qs[z_inds])
for i in z_inds
    vline!([Qs[i]], label="0", linestyle=:dot, color=:gray)
end
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")



p5 = plot(Qs, bal, lw=2, label="balance",xlim=(Qs[z_inds[1]],Qs[z_inds[2]]))#,ylim=(-40, 40))#, xlim = (-.11,-.09))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")

p2 = plot(Qs, -imag(Δs), lw=2, label="Im(Δ)",ylim=(-5,2))#,xlim=(-.11,-.09))
plot!(p2, Qs, real(Δs), lw=2, label="Re(Δ)")
#vline!([Qpeak], label="peak Q", linestyle=:dash)
#hline!([0.0], label="zero", linestyle=:dashdot)
vline!([-3], linestyle=:dashdot)
vline!([-2], linestyle=:dashdot)
vline!([Qpeak], label="peak Q", linestyle=:dash)
vline!([0.499], linestyle=:dashdot)
xlabel!("Q")
ylabel!("Δ")
title!("Torque-balance scan")

jxb = -imag(1.0 ./ (Δs .+ 1e-2))
jxb0 = -imag(1.0 ./ (Δs))
p3 = plot(Qs, jxb, lw=2, label="-Im(1 / (Δs + 1e-2))",color=:blue)#, xlim = (-.11,-.09),ylim=(-500,500))
plot!(p3, Qs, jxb0, lw=2, label="-Im(1 / Δs)",color=:red,linestyle=:dash)
xlabel!("Q")
ylabel!("~ jxb")

p4 = plot(Qs, jxb, lw=2, label="-Im(1 / Δs)",color=:blue)#, xlim = (-.11,-.09))
xlabel!("Q")
ylabel!("~ jxb")
"""
#remake plot 1 but of derivative of balance with respect to Q
dQ = Qs[2] - Qs[1]
dbal_dQ = diff(bal) ./ dQ
p3 = plot(Qs[1:end-1], dbal_dQ, lw=2, label="d(balance)/dQ",ylim=(-1000, 1000))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("d(balance)/dQ")
"""

Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=Qs[], Qmax=Qs[zin], n=10000)


plo = plot(p1, p2,p3,p4,p5, p6, layout=(6,1), size=(1000, 2000))
display(plo)


In [ ]:
Qlow = -0.1005100510051005
Qhigh =  0.49954995499549953

#Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=Qlow, Qmax=Qhigh, n=10000)

println("Qpeak = ", Qpeak)
println("maxbal = ", maximum(bal_positive))
println("maxbal = ", maximum(bal))
println("br_crit = ", brcrit)

p1 = plot(Qs, bal, lw=2, label="balance")#,ylim=(-40, 40))
z_inds = findall(i -> bal[i]*bal[i+1] < 0, 1:length(bal)-1)
println("zero crossings at Q = ", Qs[z_inds])
for i in z_inds
    vline!([Qs[i]], label="0", linestyle=:dot, color=:gray)
end
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")


p5 = plot(Qs, bal, lw=2, label="balance",ylim=(-40, 40), xlim = (-.11,-.09))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")

p2 = plot(Qs, -imag(Δs), lw=2, label="Im(Δ)",ylim=(-2,2))#,xlim=(-.11,-.09))
plot!(p2, Qs, real(Δs), lw=2, label="Re(Δ)")
#vline!([Qpeak], label="peak Q", linestyle=:dash)
#hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("Δ")
title!("Torque-balance scan")

jxb = -imag(1.0 ./ (Δs .+ 1e-2))
jxb0 = -imag(1.0 ./ (Δs))
p3 = plot(Qs, jxb, lw=2, label="-Im(1 / (Δs + 1e-2))",color=:blue, xlim = (-.11,-.09),ylim=(-500,500))
plot!(p3, Qs, jxb0, lw=2, label="-Im(1 / Δs)",color=:red,linestyle=:dash)
xlabel!("Q")
ylabel!("~ jxb")

p4 = plot(Qs, jxb, lw=2, label="-Im(1 / Δs)",color=:blue, xlim = (-.11,-.09))
xlabel!("Q")
ylabel!("~ jxb")
"""
#remake plot 1 but of derivative of balance with respect to Q
dQ = Qs[2] - Qs[1]
dbal_dQ = diff(bal) ./ dQ
p3 = plot(Qs[1:end-1], dbal_dQ, lw=2, label="d(balance)/dQ",ylim=(-1000, 1000))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("d(balance)/dQ")
"""
plo = plot(p1, p2,p3,p4,p5, layout=(5,1), size=(1000, 2000))
display(plo)


In [ ]:
#Qs = range(-5.0, 5.0, length=200)

#vals = [torque_balance_value(tb, q) for q in Qs]
"""
n=2000:
Qpeak = 0.24512256128064033
maxbal = 0.35873684252530247
br_crit = 4.668551985584323e-5

n=200 (def):
Qpeak = 0.25125628140703515
maxbal = 0.3585219071071706
br_crit = 4.667153206033017e-5
"""

#Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=-.2, Qmax=0.5, n=10000)

println("Qpeak = ", Qpeak)
println("maxbal = ", maximum(bal_positive))
println("maxbal = ", maximum(bal))
println("br_crit = ", brcrit)

p1 = plot(Qs, bal, lw=2, label="balance")#,ylim=(-40, 40))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")

p2 = plot(Qs, -imag(Δs), lw=2, label="Im(Δs)")#,ylim=(-.2,.05))#,xlim=(-.11,-.09))
plot!(p2, Qs, real(Δs), lw=2, label="Re(Δs)")
#vline!([Qpeak], label="peak Q", linestyle=:dash)
#hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("Δs")
title!("Torque-balance scan")

"""
#remake plot 1 but of derivative of balance with respect to Q
dQ = Qs[2] - Qs[1]
dbal_dQ = diff(bal) ./ dQ
p3 = plot(Qs[1:end-1], dbal_dQ, lw=2, label="d(balance)/dQ",ylim=(-1000, 1000))
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("d(balance)/dQ")
"""
plo = plot(p1, p2, layout=(2,1), size=(800, 1000))
display(plo)


In [ ]:
jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]

p1 = plot(Qs, real.(Δs), label="Re(Δ)", lw=2)
plot!(p1, Qs, imag.(Δs), label="Im(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
title!(p1, "Inner-layer Δ(Q)")

p2 = plot(Qs, jxbs, label="jxb", lw=2)
xlabel!(p2, "Q")
ylabel!(p2, "jxb")
title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
xlabel!(p3, "Q")
ylabel!(p3, "balance")
title!(p3, "2P(Q0-Q)/jxb")

plot(p1, p2, p3, layout=(3,1), size=(800, 1000))

In [ ]:
# Check torque is balanced

viscous_torque = 2*tb.P*(tb.Q0 - Qpeak)
electromagnetic_torque = -imag(1.0 / (Δs[Qpeak_ind] + 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)

# delta_n_p = 1e-2
# jxb = -imag(1.0 / (Δ + delta_n_p))
# bal = 2.0 * tb.P * (tb.Q0 - Q) / jxb
# br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))

@printf("br_crit = %.5e\n", brcrit)
@printf("viscous_torque = %.5e\n", viscous_torque)
@printf("electromagnetic_torque = %.5e\n", electromagnetic_torque)
@printf("ratio = %.10f\n", viscous_torque / electromagnetic_torque)

In [ ]:
Qs_n, bal_n, Qs_positive_n, bal_positive_n, Qpeak_n, brcrit_n, Qpeak_ind_n, Δs_n = torque_balance_scan(tb,Qmin=-3, Qmax=3, n=2)

In [ ]:
sample_number = [2, 3, 4, 5, 10,100,1000]#,10000,100000]#,1000000]
b_crit = []
# make dicts to store Qs_n and Δs_n arrays for each sample number
Qs_n_dict = Dict{Int, Vector{Float64}}()
bal_n_dict = Dict{Int, Vector{Float64}}()
for i in sample_number
    Qs_n, bal_n, Qs_positive_n, bal_positive_n, Qpeak_n, brcrit_n, Qpeak_ind_n, Δs_n = torque_balance_scan(tb,Qmin=-3, Qmax=3, n=i)
    push!(b_crit, brcrit_n)
    push!(Qs_n_dict, i => Qs_n)
    push!(bal_n_dict, i => bal_n)
    println(bal_n)
end

# add bcrit = 3.50601e-03 for n=100000 to b_crit
#push!(b_crit, 3.50601e-03)

# make a plot of b_crit vs sample_number 
p1 = plot(sample_number, b_crit,
    lw=2,
    label="b_crit",
    #xscale=:log10,
    #yscale=:log10,
    xlabel="Sample Number",
    ylabel="b_crit")

p2 = plot(
    xlabel="Q",
    ylabel="balance",
    title="Torque balance vs Q"
)

for i in sample_number[1:end-1]
    plot!(p2, Qs_n_dict[i], bal_n_dict[i],
        lw=2,
        label="n=$i")
end

p3 = plot(layout=(3, 2), size=(1000, 800))

for (k, i) in enumerate(sample_number[1:end-1])
    plot!(
        p3[k],
        Qs_n_dict[i],
        bal_n_dict[i],
        lw=2,
        label="n=$i",
        xlabel="Q",
        ylabel="balance",
        title="n = $i"
    )
end

display(p1)
display(p2)
display(p3)


println("b_crit = ", b_crit)


In [ ]:
println(b_crit)

In [ ]:
bcrit = Any[0.0014075170156654356, 0.0015190532807794722, 0.0015297173529485084, 0.0015297510027746748, 0.0015297613496804952, 0.0015297692159317363, 0.004702965572292843]

In [ ]:
using Serialization

# serialize("Qs_n_dict.jls", Qs_n_dict)
# serialize("bal_n_dict.jls", bal_n_dict)
# serialize("b_crit.jls", b_crit)
# serialize("sample_number.jls", sample_number)
#my_dict = deserialize("my_dict.jls")


In [ ]:

Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=-2.5, Qmax=0, n=20000)

jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
T_VISC = [2.0 * tb.P * (tb.Q0 - q) for q in Qs] #(q, jxb) in zip(Qs, jxbs)]
br_t = brcrit
println(tb.lu, " ", tb.sval, " ", br_t, " ", p.bt, " ", 1e-2)
@printf("br_crit = %.5e\n", brcrit)
#T_EM = tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs
T_EM = -imag(1.0 ./ (Δs .+ 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)
i = Qpeak_ind
println(T_VISC[i]/(tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs[i]))
println(T_VISC[i] / (tb.lu * tb.sval^2/2 * (brcrit * p.bt)^2 * jxbs[i]))
println(T_VISC[i] / ( (brcrit / p.bt)^2 * jxbs[i]))

#T_EM ∝ S ξ̂ (br/Bφ)² Im[-Δ̂(Q)⁻¹]
#T_visc ∝ 2 P (Q0 − Q)

xmi = -2.5
xma = 0

"""
p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2, xlim=(xmi, xma))
#plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
#title!(p1, "Inner-layer Δ(Q)")
"""

p1 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(-1000,1000))
plot!(p1, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p1, "Q")
ylabel!(p1, "Torque")

#p2 = plot(Qs, jxbs, label="jxb", lw=2)
p2 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(0,500))
plot!(p2, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p2, "Q")
ylabel!(p2, "Torque")
#title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2, xlim=(xmi, xma))
plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
ylabel!(p3, "Torque Balance")
#title!(p3, "2P(Q0-Q)/jxb")
xlabel!(p3, "Q")
vline!([Qpeak], label="Max Balance", linestyle=:dash)

plot(p1, p2, p3, layout=(3,1), size=(800, 1000))

In [ ]:
# Recreate Cihan's plot:

using Plots
using DataInterpolations

default(
    fontfamily="Georgia",
    grid=false,
    linewidth=2,
    guidefontsize=26,
    tickfontsize=18,
    titlefontsize=30,
    legendfontsize=18,
    size=(1200, 800),
    dpi=150
)

# Build the critical boundary bcrit(Q0)
Q0_scan = range(-1.0, 4.5; length=80)
bcrit_curve = Float64[]

for Q0 in Q0_scan
    tbQ = TorqueBalance(
        tb.model,
        tb.params,
        Float64(Q0),
        tb.P,
        tb.delta_n_p,
        tb.lu,
        tb.sval
    )

    Qs, bal, Qs_pos, bal_pos, Qpeak, brcrit, Qpeak_ind, Δs =
        torque_balance_scan(tbQ; Qmin=-10.0, Qmax=10.0, n=250)

    if isempty(bal_pos) || !all(isfinite.(bal_pos)) || maximum(bal_pos) <= 0
        push!(bcrit_curve, NaN)
    else
        push!(bcrit_curve, brcrit)
    end
end

ok = isfinite.(bcrit_curve)
Q0_plot = Q0_scan[ok]
bcrit_plot = bcrit_curve[ok]

perm = sortperm(Q0_plot)
Q0_plot = Q0_plot[perm]
bcrit_plot = bcrit_plot[perm]

# Interpolate the boundary: bcrit = f(Q0)
curve_itp = LinearInterpolation(bcrit_plot, Q0_plot)

# Full phase-box bounds
x0 = minimum(bcrit_plot) * 0.8
x1 = maximum(bcrit_plot) * 1.2
y0 = minimum(Q0_plot)
y1 = maximum(Q0_plot)

xvals = range(x0, x1; length=350)
yvals = range(y0, y1; length=350)

locked_mask = zeros(Float64, length(yvals), length(xvals))
for iy in eachindex(yvals)
    y = yvals[iy]
    for ix in eachindex(xvals)
        x = xvals[ix]
        # locked when bcrit exceeds the boundary for that Q0
        locked_mask[iy, ix] = x >= curve_itp(y) ? 1.0 : 0.0
    end
end

p = contourf(
    xvals, yvals, locked_mask',
    levels=[-0.5, 0.5, 1.5],
    c=[:navy, :firebrick],
    colorbar=false,
    xlabel="b_crit",
    ylabel="Q_0",
    title="Locked / unlocked region",
    framestyle=:box,
    xlims=(x0, x1),
    ylims=(y0, y1),
    legend=false
)

# Critical boundary
plot!(p, bcrit_plot, Q0_plot; lw=3, color=:black, label="critical boundary")

# Critical point
cp_idx = argmax(bcrit_plot)
cp_x = bcrit_plot[cp_idx]
cp_y = Q0_plot[cp_idx]
scatter!(p, [cp_x], [cp_y]; color=:gold, marker=:star5, ms=12, label="critical point")

# Single legend block with one entry per category
plot!(p, [NaN], [NaN]; color=:firebrick, lw=4, label="locked")
plot!(p, [NaN], [NaN]; color=:navy, lw=4, label="unlocked")
plot!(p, [NaN], [NaN]; color=:black, lw=3, label="critical boundary")
scatter!(p, [NaN], [NaN]; color=:gold, marker=:star5, ms=12, label="critical point")

plot!(p; legend=:topright)

display(p)